In [15]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

print("Model ready!")

Model ready!


In [16]:
from langchain.tools import tool

_fetch_attempts = {"count": 0}

@tool
def fetch_live_external_data(topic: str) -> str:
    """Fetch live or real-time external data for a topic (e.g. weather, stock quote, breaking news).

    Use this when the user asks for up-to-the-minute or live information.
    """
    _fetch_attempts["count"] += 1
    attempt = _fetch_attempts["count"]
    print(f"fetch_live_external_data call #{attempt} for topic={topic!r}")

    # Demo: fail the first two times, succeed on the third so retry middleware can recover
    if attempt <= 2:
        raise ConnectionError("Simulated failure: external data service is unreachable.")

    return (
        f"Live data for '{topic}': Bengaluru (Bangalore) is currently 27°C, "
        "partly cloudy, with a light breeze from the east."
    )

In [17]:
from langchain.agents.middleware import wrap_model_call, ModelRequest , wrap_tool_call, ToolCallRequest
import time
from langchain_core.messages import ToolMessage

@wrap_tool_call
def tool_retry_middleware(request: ToolCallRequest, handler):

    max_attempts = 3
    retry_delay_seconds = 2
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            print(f"Tool call attempt {attempt}/{max_attempts}")
            return handler(request)
        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                print(f"Retrying in {retry_delay_seconds} seconds...")
                time.sleep(retry_delay_seconds)

    msg = str(last_error).lower()
    if "connection" in msg or "timeout" in msg or "unreachable" in msg:
        return ToolMessage(
            content=(
                "The live data service is temporarily unavailable after 3 attempts. "
                "Answer from general knowledge or suggest the user try again later."
            ),
            tool_call_id=request.tool_call["id"],
        )
    return ToolMessage(
        content=f"That tool hit an error after 3 attempts: {last_error!s}. Please rephrase or use another approach.",
        tool_call_id=request.tool_call["id"],
    )




@wrap_model_call
def logging_middleware(request: ModelRequest, handler) :

    messages=request.state['messages']
    last_message=messages[-1].content

    print(f"Last message: {last_message}")   
    print(f"Model: {request.model}")

    response=handler(request)

    print(f"Response: {response}")

    return response

@wrap_model_call
def dynamicmodel_middleware(request: ModelRequest, handler):

    last_msg=request.state['messages'][-1].content

    complex_keywords = ["complex", "multiple", "detailed", "analysis", "explain"]

    if any(keyword in last_msg.lower() for keyword in complex_keywords):
        model=ChatOpenAI(model="gpt-5.4-mini", temperature=0)
        print("Using complex model")
    else:
        model=ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        print("Using simple model")

    request.override(model=model)

    response=handler(request)

    return response




In [18]:
from langchain_tavily import TavilySearch
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
    model = model,
    tools=[TavilySearch(), fetch_live_external_data],
    middleware=[ dynamicmodel_middleware, logging_middleware ,
     ToolRetryMiddleware(
        max_retries=3,
        initial_delay=2,
        backoff_factor=2
    )]
 )

result =agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "what is the temperature in Bengaluru"}
        ]
    }
 )


result

Using simple model
Last message: what is the temperature in Bengaluru
Model: metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2', 'langchain-openai': '1.6.6'}} profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000020AFF216300> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020AFF

{'messages': [HumanMessage(content='what is the temperature in Bengaluru', additional_kwargs={}, response_metadata={}, id='bc0f16c0-cfe4-4673-8090-4cd87267d3b4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 1247, 'total_tokens': 1265, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1152, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_89f6f120c4', 'id': 'chatcmpl-ETlyHlNyDkbC9hyUNv4dVdtg2Z0X0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f1e3-97b8-7d02-a7c8-0372f2dc3d2c-0', tool_calls=[{'name': 'fetch_live_external_data', 'args': {'topic': 'temperature in B

In [ ]:
from re import A
from langchain.agents.middleware import  PIIMiddleware

agent = create_agent(
    model = model,
    tools=[TavilySearch()],
    middleware=[
        PIIMiddleware(
            pii_type="aadhar",
            strategy="mask",
            detector=r"(\d{4}-\d{4}-\d{4})",
            apply_to_input=True,
            apply_to_output=True,
            
        ), 
         PIIMiddleware(
            pii_type="credit_card",
            strategy="mask",
           
            apply_to_input=True,
            apply_to_output=True,
            
        )
    ]
 )

result = agent.invoke(
    {
        "messages": [
                {"role": "user", "content": "my credit card number is 4234-5678-9012  check if i am eligible for a discount"}
            ]
    }
)

result


{'messages': [HumanMessage(content='my credit card number is ****9012  check if i am eligible for a discount', additional_kwargs={}, response_metadata={}, id='9c7c41d0-4734-4535-a588-d620969a88a8'),
  AIMessage(content="I'm sorry, but I can't process or check eligibility based on credit card numbers. If you provide more details about the discount or the program you're referring to, I can help you find the relevant information or guide you on how to check your eligibility.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 1204, 'total_tokens': 1254, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fing

In [34]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, InterruptOnConfig
from  langgraph.checkpoint.memory import  InMemorySaver

@tool
def approve_payment(amount: float) -> str:

    """Approve a payment of a given amount"""

    print(f"Approving payment of {amount} dollars")

    return f"Payment of {amount} dollars approved"


agent = create_agent(
    model = model,
    tools=[approve_payment],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "approve_payment": InterruptOnConfig(
                                allowed_decisions=["approve", "reject"],
                                when=lambda req: req.tool_call["args"].get("amount", "") >50000,
                            )
        }
    )]
)  

config = {"configurable": {"thread_id": "1"}}

result= agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "approve a payment of 60000 dollars"}
        ]
    },
    config=config
)

result

{'messages': [HumanMessage(content='approve a payment of 60000 dollars', additional_kwargs={}, response_metadata={}, id='81b2a1de-255d-488e-b073-85c65b69a36a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 54, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1d66328c7d', 'id': 'chatcmpl-ETmKabJyUahjcuyRy5D8Odo0Sf0rD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f1f8-b54e-74b1-afa9-5ff3cfc9a139-0', tool_calls=[{'name': 'approve_payment', 'args': {'amount': 60000}, 'id': 'call_OhWMW8mjMfALkY

In [30]:
result['__interrupt__'][0].value

{'action_requests': [{'name': 'approve_payment',
   'args': {'amount': 1000},
   'description': "Tool execution requires approval\n\nTool: approve_payment\nArgs: {'amount': 1000}"}],
 'review_configs': [{'action_name': 'approve_payment',
   'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}

In [27]:
result['messages'][1].tool_calls

[{'name': 'approve_payment',
  'args': {'amount': 1000},
  'id': 'call_Nlf6QIIs1vwVaTtiW6AXJox3',
  'type': 'tool_call'}]

In [35]:
from langgraph.types import Command

result =agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]} ),
    config=config
)

result

{'messages': [HumanMessage(content='approve a payment of 60000 dollars', additional_kwargs={}, response_metadata={}, id='81b2a1de-255d-488e-b073-85c65b69a36a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 54, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_1d66328c7d', 'id': 'chatcmpl-ETmKabJyUahjcuyRy5D8Odo0Sf0rD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f1f8-b54e-74b1-afa9-5ff3cfc9a139-0', tool_calls=[{'name': 'approve_payment', 'args': {'amount': 60000}, 'id': 'call_OhWMW8mjMfALkY